In [ ]:
# %%
# LoopNav rollout: retrieval arms on one loop clip. Machinery lives in
# far.analysis.loopnav_rollout (R.*); this notebook is knobs + calls.
# Every arm is its OWN training run (the same cdit_b pose-conditioned generator,
# a different memory) rolled along the SAME GT return path: replace-tail protocol
# (the pool's history stays 100% real), poses re-centred on the new current frame
# every step exactly as the training formatter does, same noise seed for every arm.
#
# Needs: the checkpoint bundles under models/loopnav/<arm>/ (each carries its
# config.yaml and checkpoints/<step>.pth.tar), the pre-computed latents under
# datasets/loopnav/latent/, and the manifest / test indices under
# results/manifests/loopnav/ and results/indices/loopnav/. The GPU is chosen by
# the FAR_DEVICE environment variable (default cuda:0).
import os
import numpy as np
import torch
from IPython.display import Video, display

from far.constants import ROOT
import far.analysis.loopnav_rollout as R

os.chdir(ROOT)
os.environ.setdefault("FAR_MODELS_ROOT", os.path.join(ROOT, "models"))
os.environ.setdefault("FAR_RESULT_ROOT", os.path.join(ROOT, "results"))
device = os.environ.get("FAR_DEVICE", "cuda:0") if torch.cuda.is_available() else "cpu"

# label -> (run dir, checkpoint step); each run dir carries its config. All runs must
# share ONE pose convention (build_models asserts it); the LAST entry's config builds
# the shared dataset, so put a FAR arm last when several arms are enabled. Keys are the
# DISPLAY NAMES shown on every figure / video; the commented entries are the other
# paper arms (FAR -- Meta, the ablations, LongLive-RAG, the heuristic baselines).
RUNS = {
    "FAR":               ("models/loopnav/far_multicue", "0700000"),
    # "Temporal":          ("models/loopnav/temporal", "0700000"),
    # "WorldMem":          ("models/loopnav/worldmem", "0700000"),
    # "FAR (meta only)":   ("models/loopnav/far_meta", "0300000"),
    # "FAR (frozen retriever)": ("models/loopnav/far_frozen_encoder", "0700000"),
    # "FAR (visual only)": ("models/loopnav/far_visual", "0300000"),
    # "LongLive-RAG":      ("models/loopnav/longlive_rag", "0700000"),
}
print("device:", device)


In [ ]:
# %%
# Build the arms and the RAW test dataset (global poses kept for the per-step
# re-centring; latents pre-computed by the LatentLoopNavBackend).
POOL_SIZE     = 1024                   # retrieval pool; covers the whole outbound leg
DATASET_GROUP = "loopnav_latent"

models, data_cfg = R.build_models(RUNS, device)
dataset, spec, stats_np, STANDARD_FRAME = R.build_dataset(data_cfg, DATASET_GROUP, POOL_SIZE)


In [ ]:
# %%
# Pick the clip and the rollout window — the only cell to re-run per clip.
CLIP_IDX       = 12   # test clip
SEED           = 0      # worldmem-subsample seed; also re-seeds rollout noise
ROLLOUT_STRIDE = 8      # GT frames per generation step (fps 20 -> 0.1 s/step)
START_OFFSET, MAX_STEPS = 150, 100  # the window: frames into the return path, then a step cap (None = to its end)
POOL_PHASE     = "exploration"   # "traversed" = walked return-path frames join the retrievable pool | "exploration" = they do not,
                               #   so every retrievable frame predates the return (memory of the outbound leg alone)
SAMPLER, SAMPLING_STEPS = "ddim", 20
PNP_CHUNK_SIZE = 10      # None = each run's trained value
CONTEXT_SIZE   = 12     # None = trained value (4 -> 3 memories)
WORLDMEM_POOL_SIZE = 200   # worldmem-only pool cap (matched to training)
INSERT_GENERATED = False   # True = generated frames become retrievable history
TEACHER_FORCED = None   # None | "query" (GT frame for the retrieval query only) | "full" (GT pool tail: the upper bound)

SAVE_DIR = f"results/rollout_video_loopnav/{CLIP_IDX}"; os.makedirs(SAVE_DIR, exist_ok=True)
R.install_eval_knobs(models, data_cfg, SAMPLER, SAMPLING_STEPS, PNP_CHUNK_SIZE, CONTEXT_SIZE)
ep = R.load_episode(dataset, CLIP_IDX, SEED, standard_frame=STANDARD_FRAME)
targets = R.apply_window(ep, spec, START_OFFSET, ROLLOUT_STRIDE, MAX_STEPS, POOL_PHASE)
decode = R.Decoder(next(iter(models.values())).tokenizer, device)
R.inspect_trajectory(ep, spec, decode)


In [ ]:
# %%
# Full-episode video: the clip from its first frame to its last, GROUND TRUTH ONLY, in the rollout video's
# style — one tight panel: "History / evaluation in N s" over the OUTBOUND leg, the transition card, then
# "Ground truth / evaluation" over the return path. Every frame comes from the clip's latent .npy at
# ROLLOUT_STRIDE spacing.
HISTORY_SPEED, EVAL_SPEED = 1, 1   # playback speed of each phase, x real time (any positive value, e.g. 1.5 or 0.5)
import far.analysis.thor_rollout as TR
_ep_t = R.thor_episode(ep, ROLLOUT_STRIDE)
TR.write_rollout_video(_ep_t, spec, decode, decode(_ep_t["goal_lat"][_ep_t["targets"]]), {}, {}, "{}", "",
                       np.zeros((len(_ep_t["targets"]), 4), int), {}, {}, f"{SAVE_DIR}/full_episode.mp4",
                       include_interaction_phase=True, history_speed=HISTORY_SPEED, eval_speed=EVAL_SPEED, width=800, outer_pad=0)


In [ ]:
# %%
# Roll every arm on the same window (same noise seed); hooks record each arm's picks
# (+ scores) and, for fused arms, channel attribution.
preds, score_stats, attribution, pick_times, pick_scores = R.run_rollouts(
    models, ep, spec, stats_np, device, sampler=SAMPLER, seed=SEED,
    insert_generated=INSERT_GENERATED, teacher_forced=TEACHER_FORCED,
    worldmem_pool_size  =WORLDMEM_POOL_SIZE)
gt_lat = ep["goal_lat"][targets]


In [ ]:
# %%
# Decode + metrics + the two summary figures.
gt_img   = decode(gt_lat)
pred_img = {l: decode(p) for l, p in preds.items()}
cur_img  = decode(ep["ctx_lat"][-1:])[0]
psnr, lpips_steps, dreamsim_steps = R.compute_metrics(pred_img, gt_img, device)
_tf = "" if TEACHER_FORCED is None else f", TEACHER-FORCED ({TEACHER_FORCED})"
R.plot_compare(ep, spec, cur_img, gt_img, pred_img, psnr,
               title_extra=f", stride {ROLLOUT_STRIDE}, {SAMPLER}{SAMPLING_STEPS}{_tf}",
               save_path=f"{SAVE_DIR}/frames_comparison.png")
R.plot_summary(ep, spec, psnr, lpips_steps, dreamsim_steps, save_path=f"{SAVE_DIR}/summary_metrics.png")


In [ ]:
# %%
# Rollout video in the ai2thor style (far.analysis.thor_rollout.write_rollout_video): every row a panel with a
# name card and a coloured edge; the OUTBOUND leg plays first as "History / evaluation in N s" (GT only), a
# transition card marks the return path, then "Ground truth" vs each arm's prediction beside the memories it
# recalled ("Context #k from -t sec", best-ranked first). Navigation only: no interaction captions / verdicts.
HISTORY_SPEED, EVAL_SPEED = 1.0, 1.0     # playback speed of the outbound leg / the rollout, x real time (any positive value)
ERROR_OVERLAY = True                 # red ERROR PANES on each arm's prediction tile:
ERROR_THRESH, ERROR_PATCH, ERROR_ALPHA = 0.15, 32, 0.3   #   patch x patch squares (256-px units) tinted at alpha where mean |pred - GT| > thresh
OURS = "FAR"                         # the arm drawn as ours (amber name + "(ours)" line); a RUNS label, None = none
NAMES = R.DISPLAY_NAME               # RUNS label -> display name on the cards
import far.analysis.thor_rollout as TR
ep_t = R.thor_episode(ep, ROLLOUT_STRIDE)
TR.write_rollout_video(ep_t, spec, decode, gt_img, {NAMES.get(l, l): v for l, v in pred_img.items()}, lpips_steps, "{:.3f}", "",
                       np.zeros((gt_img.shape[0], 4), int), {NAMES.get(l, l): v for l, v in pick_times.items()},
                       {NAMES.get(l, l): v for l, v in pick_scores.items()}, f"{SAVE_DIR}/rollout_panels.mp4", pick_top_n=4,
                       include_interaction_phase=True, history_speed=HISTORY_SPEED, eval_speed=EVAL_SPEED,
                       error_overlay=ERROR_OVERLAY, error_thresh=ERROR_THRESH, error_patch=ERROR_PATCH, error_alpha=ERROR_ALPHA,
                       ours=NAMES.get(OURS, OURS), device=device)


In [ ]:
# %%
# Static frame progression (video conventions: display names + error overlay).
stem = os.path.basename(ep["video_id"]).replace(".npy", "")
_tftag = "" if TEACHER_FORCED is None else f"_tf-{TEACHER_FORCED}"
R.plot_progression(ep, spec, cur_img, gt_img, pred_img, lpips_steps, display_name=R.DISPLAY_NAME,
                   save_path=f"{SAVE_DIR}/clip{CLIP_IDX:05d}_{stem}_off{START_OFFSET}_{SAMPLER}{SAMPLING_STEPS}{_tftag}.png")


In [ ]:
# %%
# Retrieval geography: WHERE each arm reads from, per step (scrubbable).
geo_path = R.write_geography(ep, spec, pick_times, pick_scores, gt_img, pred_img, psnr, decode,
                             f"{SAVE_DIR}/retrieval_geography.mp4")
display(Video(geo_path, embed=True, width=960))


In [ ]:
# %%
# Rollout video: GT vs arms + evolving LPIPS strip (R.DISPLAY_NAME relabels panels;
# error-overlay knobs are write_rollout_video kwargs).
video_path = R.write_rollout_video(
    ep, spec, cur_img, gt_img, pred_img, lpips_steps,
    f"{SAVE_DIR}/clip{CLIP_IDX:05d}_{stem}_off{START_OFFSET}_{SAMPLER}{SAMPLING_STEPS}{_tftag}.mp4",
    rollout_stride=ROLLOUT_STRIDE, display_name=R.DISPLAY_NAME)
display(Video(video_path, embed=True, width=960))


In [ ]:
# %%
# Save the frames as PNGs for figures / eyeballing: the GT frame at each rollout step, each arm's
# generated frame, and the TOP-1 context frame the arm retrieved for that step.
ALL_GT = False   # also dump every GT frame of the return path (not just the rollout targets).
                 #   NOTE it dumps what the episode dict holds, so under START_OFFSET > 0 the
                 #   frames walked BEFORE the window are absent — they were sliced into the pool.

R.export_frames(ep, spec, decode, gt_img, pred_img, cur_img, pick_times, SAVE_DIR, all_gt=ALL_GT)
# ...and the numbers that go with them -> SAVE_DIR/meta.json: per step, per arm, the metric,
# the retrieved pick times AND those picks as episode frame indices (the frames themselves are
# in frames/gt/ when all_gt was on).
R.export_records(ep, spec, targets, pred_img, pick_times, SAVE_DIR,
                 metrics={"lpips": lpips_steps, "psnr": psnr}, clip_idx=CLIP_IDX, seed=SEED,
                 sampler=SAMPLER, sampling_steps=SAMPLING_STEPS, pool_size=POOL_SIZE,
                 context_size=CONTEXT_SIZE, pnp_chunk_size=PNP_CHUNK_SIZE, start_offset=START_OFFSET,
                 runs={l: list(RUNS[l]) for l in pred_img if l in RUNS})
